In [ ]:
from pathlib import Path
from datetime import date

import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PANEL_FILE = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "09_sp500_monthly_accounting_panel_2003_2025.parquet"
)

RAW_FACTOR_PARQUET_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "10_sp500_monthly_raw_factors_2005_2025.parquet"
)

RAW_FACTOR_CSV_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "10_sp500_monthly_raw_factors_2005_2025.csv"
)

RAW_FACTOR_PARQUET_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

panel_df = pl.read_parquet(PANEL_FILE)

print("Monthly accounting panel loaded successfully.")
print(f"Number of rows: {panel_df.height:,}")
print(f"Number of columns: {panel_df.width}")
print(f"Panel file: {PANEL_FILE}")

In [ ]:
panel_check = panel_df.select([
    pl.col("month").min().alias("start_month"),
    pl.col("month").max().alias("end_month"),
    pl.col("permno").n_unique().alias("unique_permnos"),
    pl.col("gvkey").n_unique().alias("unique_gvkeys"),
    pl.col("monthly_return").null_count().alias("missing_returns"),
    pl.col("has_current_accounting_data")
    .sum()
    .alias("rows_with_current_accounting_data"),
])

print("Input panel summary:")
print(panel_check)

In [ ]:
factor_df = (
    panel_df
    .sort([
        "permno",
        "month",
    ])
    .with_columns([
        (
            pl.col("month_end_market_cap")
            / 1000.0
        ).alias("market_cap_usd_millions"),

        (
            pl.col("month").dt.year() * 12
            + pl.col("month").dt.month()
        ).alias("_month_index"),
    ])
)

factor_df = factor_df.with_columns([
    # Book-to-Market
    pl.when(
        pl.col("has_current_accounting_data")
        & (pl.col("book_equity") > 0)
        & (pl.col("market_cap_usd_millions") > 0)
    )
    .then(
        pl.col("book_equity")
        / pl.col("market_cap_usd_millions")
    )
    .otherwise(None)
    .alias("book_to_market"),

    # Earnings-to-Price
    pl.when(
        pl.col("has_current_accounting_data")
        & pl.col("earnings").is_not_null()
        & (pl.col("market_cap_usd_millions") > 0)
    )
    .then(
        pl.col("earnings")
        / pl.col("market_cap_usd_millions")
    )
    .otherwise(None)
    .alias("earnings_to_price"),

    # Size
    pl.when(
        pl.col("market_cap_usd_millions") > 0
    )
    .then(
        pl.col("market_cap_usd_millions").log()
    )
    .otherwise(None)
    .alias("log_market_cap"),

    # Quality components
    pl.when(
        pl.col("has_current_accounting_data")
    )
    .then(pl.col("operating_profitability"))
    .otherwise(None)
    .alias("quality_profitability_raw"),

    pl.when(
        pl.col("has_current_accounting_data")
    )
    .then(pl.col("roe"))
    .otherwise(None)
    .alias("quality_roe_raw"),

    pl.when(
        pl.col("has_current_accounting_data")
    )
    .then(pl.col("accruals"))
    .otherwise(None)
    .alias("quality_accruals_raw"),

    # Investment
    pl.when(
        pl.col("has_current_accounting_data")
    )
    .then(pl.col("asset_growth"))
    .otherwise(None)
    .alias("investment_growth_raw"),
])

print("Value, size, quality, and investment inputs were created.")

In [ ]:
factor_df = factor_df.with_columns(
    pl.when(
        pl.col("monthly_return").is_not_null()
        & (pl.col("monthly_return") > -1.0)
    )
    .then(
        (
            1.0
            + pl.col("monthly_return")
        ).log()
    )
    .otherwise(None)
    .alias("_log_monthly_return")
)

factor_df = factor_df.with_columns([
    pl.col("_log_monthly_return")
    .shift(1)
    .over("permno")
    .alias("_lagged_log_return"),

    pl.col("month")
    .shift(11)
    .over("permno")
    .alias("_month_lag_11"),

    pl.when(
        pl.col("number_of_return_observations") > 1
    )
    .then(
        (
            pl.col("number_of_return_observations")
            - 1
        ).cast(pl.Float64)
    )
    .otherwise(0.0)
    .alias("_volatility_weight"),
])

In [ ]:
factor_df = factor_df.with_columns(
    (
        pl.col("_volatility_weight")
        * pl.col("realized_volatility_annualized").pow(2)
    ).alias("_weighted_volatility_squared")
)

In [ ]:
factor_df = factor_df.with_columns([
    pl.col("_lagged_log_return")
    .rolling_sum(
        window_size=11,
        min_periods=11,
    )
    .over("permno")
    .alias("_momentum_log_sum"),

    pl.col("_weighted_volatility_squared")
    .rolling_sum(
        window_size=12,
        min_periods=12,
    )
    .over("permno")
    .alias("_volatility_numerator_12m"),

    pl.col("_volatility_weight")
    .rolling_sum(
        window_size=12,
        min_periods=12,
    )
    .over("permno")
    .alias("_volatility_denominator_12m"),
])

In [ ]:
factor_df = factor_df.with_columns(
    (
        pl.col("_month_lag_11").is_not_null()
        & (
            pl.col("_month_index")
            - (
                pl.col("_month_lag_11").dt.year() * 12
                + pl.col("_month_lag_11").dt.month()
            )
            == 11
        )
    ).alias("_has_continuous_12m_history")
)

In [ ]:
factor_df = factor_df.with_columns([
    pl.when(
        pl.col("_has_continuous_12m_history")
        & pl.col("_momentum_log_sum").is_not_null()
    )
    .then(
        pl.col("_momentum_log_sum").exp() - 1.0
    )
    .otherwise(None)
    .alias("momentum_12_1"),

    pl.when(
        pl.col("_has_continuous_12m_history")
        & (
            pl.col("_volatility_denominator_12m")
            > 0
        )
    )
    .then(
        (
            pl.col("_volatility_numerator_12m")
            / pl.col("_volatility_denominator_12m")
        ).sqrt()
    )
    .otherwise(None)
    .alias("volatility_12m"),
])

print("Momentum and low-volatility signals were created.")

In [ ]:
factor_df = factor_df.with_columns([
    pl.col("_month_index")
    .shift(-1)
    .over("permno")
    .alias("_next_month_index"),

    pl.col("monthly_return")
    .shift(-1)
    .over("permno")
    .alias("_next_monthly_return"),
])

factor_df = factor_df.with_columns(
    pl.when(
        pl.col("_next_month_index").is_not_null()
        & (
            pl.col("_next_month_index")
            - pl.col("_month_index")
            == 1
        )
    )
    .then(
        pl.col("_next_monthly_return")
    )
    .otherwise(None)
    .alias("future_return_1m")
)

print("One-month-ahead returns were created.")

In [ ]:
helper_columns = [
    "_month_index",
    "_log_monthly_return",
    "_lagged_log_return",
    "_month_lag_11",
    "_volatility_weight",
    "_weighted_volatility_squared",
    "_momentum_log_sum",
    "_volatility_numerator_12m",
    "_volatility_denominator_12m",
    "_has_continuous_12m_history",
    "_next_month_index",
    "_next_monthly_return",
]

raw_factor_df = (
    factor_df
    .filter(
        (pl.col("month") >= date(2005, 1, 31))
        & pl.col("is_member_month_end")
    )
    .drop(helper_columns)
    .sort([
        "month",
        "permno",
    ])
)

print("Raw factor sample created successfully.")
print(f"Number of rows: {raw_factor_df.height:,}")
print(f"Number of columns: {raw_factor_df.width}")

In [ ]:
raw_signal_columns = [
    "book_to_market",
    "earnings_to_price",
    "momentum_12_1",
    "quality_profitability_raw",
    "quality_roe_raw",
    "quality_accruals_raw",
    "investment_growth_raw",
    "log_market_cap",
    "volatility_12m",
    "future_return_1m",
]

signal_coverage = raw_factor_df.select([
    pl.len().alias("total_rows"),

    *[
        pl.col(column)
        .is_not_null()
        .sum()
        .alias(f"{column}_available")
        for column in raw_signal_columns
    ],
])

print("Raw factor availability summary:")
print(signal_coverage)

In [ ]:
coverage_counts = raw_factor_df.select([
    pl.col(column)
    .is_not_null()
    .sum()
    .alias(column)
    for column in raw_signal_columns
]).row(0)

print("Raw factor coverage rates:")

for column, available_count in zip(
    raw_signal_columns,
    coverage_counts,
):
    coverage_rate = available_count / raw_factor_df.height

    print(
        f"{column}: "
        f"{available_count:,} "
        f"({coverage_rate:.2%})"
    )

raw_factor_df.write_parquet(
    RAW_FACTOR_PARQUET_FILE,
    compression="zstd",
)

raw_factor_df.write_csv(
    RAW_FACTOR_CSV_FILE
)

print("\nRaw factor datasets saved successfully.")
print(f"Parquet file: {RAW_FACTOR_PARQUET_FILE}")
print(f"CSV file: {RAW_FACTOR_CSV_FILE}")

In [ ]:
signal_direction_map = {
    "book_to_market": (
        "z_book_to_market",
        1.0,
    ),
    "earnings_to_price": (
        "z_earnings_to_price",
        1.0,
    ),
    "momentum_12_1": (
        "z_momentum_12_1",
        1.0,
    ),
    "quality_profitability_raw": (
        "z_quality_profitability",
        1.0,
    ),
    "quality_roe_raw": (
        "z_quality_roe",
        1.0,
    ),
    "quality_accruals_raw": (
        "z_quality_accruals",
        -1.0,
    ),
    "investment_growth_raw": (
        "z_investment",
        -1.0,
    ),
    "log_market_cap": (
        "z_size",
        -1.0,
    ),
    "volatility_12m": (
        "z_low_volatility",
        -1.0,
    ),
}

raw_standardization_columns = list(
    signal_direction_map.keys()
)

print("Factor direction definitions created.")

In [ ]:
standardized_df = raw_factor_df.with_columns([
    pl.when(
        pl.col(column).is_finite()
    )
    .then(pl.col(column))
    .otherwise(None)
    .alias(column)
    for column in raw_standardization_columns
])

In [ ]:
winsor_expressions = []

for column in raw_standardization_columns:
    lower_bound = (
        pl.col(column)
        .quantile(0.01)
        .over("month")
    )

    upper_bound = (
        pl.col(column)
        .quantile(0.99)
        .over("month")
    )

    winsor_expression = (
        pl.when(
            pl.col(column) < lower_bound
        )
        .then(lower_bound)
        .when(
            pl.col(column) > upper_bound
        )
        .then(upper_bound)
        .otherwise(
            pl.col(column)
        )
        .alias(f"{column}_winsorized")
    )

    winsor_expressions.append(
        winsor_expression
    )

standardized_df = standardized_df.with_columns(
    winsor_expressions
)

print("Monthly 1st–99th percentile winsorization completed.")

In [ ]:
zscore_expressions = []

for raw_column, (
    zscore_column,
    direction,
) in signal_direction_map.items():

    winsorized_column = (
        f"{raw_column}_winsorized"
    )

    monthly_mean = (
        pl.col(winsorized_column)
        .mean()
        .over("month")
    )

    monthly_std = (
        pl.col(winsorized_column)
        .std(ddof=1)
        .over("month")
    )

    zscore_expression = (
        pl.when(
            monthly_std.is_not_null()
            & (monthly_std > 0)
        )
        .then(
            direction
            * (
                pl.col(winsorized_column)
                - monthly_mean
            )
            / monthly_std
        )
        .otherwise(None)
        .alias(zscore_column)
    )

    zscore_expressions.append(
        zscore_expression
    )

standardized_df = standardized_df.with_columns(
    zscore_expressions
)

print("Monthly cross-sectional z-scores were created.")

In [ ]:
standardized_df = standardized_df.with_columns([
    pl.sum_horizontal([
        pl.col("z_book_to_market")
        .is_not_null()
        .cast(pl.Int8),

        pl.col("z_earnings_to_price")
        .is_not_null()
        .cast(pl.Int8),
    ]).alias("value_component_count"),

    pl.sum_horizontal([
        pl.col("z_quality_profitability")
        .is_not_null()
        .cast(pl.Int8),

        pl.col("z_quality_roe")
        .is_not_null()
        .cast(pl.Int8),

        pl.col("z_quality_accruals")
        .is_not_null()
        .cast(pl.Int8),
    ]).alias("quality_component_count"),
])

In [ ]:
standardized_df = standardized_df.with_columns([
    pl.when(
        pl.col("value_component_count") >= 1
    )
    .then(
        pl.mean_horizontal([
            "z_book_to_market",
            "z_earnings_to_price",
        ])
    )
    .otherwise(None)
    .alias("_factor_value_raw"),

    pl.col("z_momentum_12_1")
    .alias("_factor_momentum_raw"),

    pl.when(
        pl.col("quality_component_count") >= 2
    )
    .then(
        pl.mean_horizontal([
            "z_quality_profitability",
            "z_quality_roe",
            "z_quality_accruals",
        ])
    )
    .otherwise(None)
    .alias("_factor_quality_raw"),

    pl.col("z_investment")
    .alias("_factor_investment_raw"),

    pl.col("z_size")
    .alias("_factor_size_raw"),

    pl.col("z_low_volatility")
    .alias("_factor_low_volatility_raw"),
])

print("Six raw factor-family scores were created.")

In [ ]:
factor_family_map = {
    "_factor_value_raw": "factor_value",
    "_factor_momentum_raw": "factor_momentum",
    "_factor_quality_raw": "factor_quality",
    "_factor_investment_raw": "factor_investment",
    "_factor_size_raw": "factor_size",
    "_factor_low_volatility_raw": "factor_low_volatility",
}

family_zscore_expressions = []

for raw_family, final_family in factor_family_map.items():
    family_mean = (
        pl.col(raw_family)
        .mean()
        .over("month")
    )

    family_std = (
        pl.col(raw_family)
        .std(ddof=1)
        .over("month")
    )

    family_zscore_expressions.append(
        pl.when(
            family_std.is_not_null()
            & (family_std > 0)
        )
        .then(
            (
                pl.col(raw_family)
                - family_mean
            )
            / family_std
        )
        .otherwise(None)
        .alias(final_family)
    )

standardized_df = standardized_df.with_columns(
    family_zscore_expressions
)

print("Six standardized factor-family scores were created.")

In [ ]:
factor_family_columns = [
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility",
]

standardized_df = standardized_df.with_columns(
    pl.sum_horizontal([
        pl.col(column)
        .is_not_null()
        .cast(pl.Int8)
        for column in factor_family_columns
    ]).alias("factor_family_count")
)

standardized_df = standardized_df.with_columns(
    pl.when(
        pl.col("factor_family_count") >= 4
    )
    .then(
        pl.mean_horizontal(
            factor_family_columns
        )
    )
    .otherwise(None)
    .alias("_multi_factor_score_raw")
)

In [ ]:
multi_factor_mean = (
    pl.col("_multi_factor_score_raw")
    .mean()
    .over("month")
)

multi_factor_std = (
    pl.col("_multi_factor_score_raw")
    .std(ddof=1)
    .over("month")
)

factor_score_df = standardized_df.with_columns(
    pl.when(
        multi_factor_std.is_not_null()
        & (multi_factor_std > 0)
    )
    .then(
        (
            pl.col("_multi_factor_score_raw")
            - multi_factor_mean
        )
        / multi_factor_std
    )
    .otherwise(None)
    .alias("multi_factor_score")
)

print("Equal-weight multi-factor scores were created.")

In [ ]:
factor_score_columns = [
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility",
    "multi_factor_score",
]

factor_score_validation = factor_score_df.select([
    *[
        pl.col(column)
        .is_not_null()
        .sum()
        .alias(f"{column}_available")
        for column in factor_score_columns
    ],
])

factor_distribution_validation = factor_score_df.select([
    *[
        pl.col(column)
        .mean()
        .alias(f"{column}_mean")
        for column in factor_score_columns
    ],

    *[
        pl.col(column)
        .std(ddof=1)
        .alias(f"{column}_std")
        for column in factor_score_columns
    ],
])

print("Factor-score availability:")
print(factor_score_validation)

print("\nFactor-score distribution summary:")
print(factor_distribution_validation)

In [ ]:
STANDARDIZED_FACTOR_PARQUET_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "11_sp500_monthly_factor_scores_full_2005_2025.parquet"
)

COMPACT_FACTOR_CSV_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "11_sp500_monthly_factor_scores_2005_2025.csv"
)

factor_score_df.write_parquet(
    STANDARDIZED_FACTOR_PARQUET_FILE,
    compression="zstd",
)

print("Full factor dataset saved successfully.")
print(f"Number of rows: {factor_score_df.height:,}")
print(f"Number of columns: {factor_score_df.width}")
print(f"Parquet file: {STANDARDIZED_FACTOR_PARQUET_FILE}")

In [ ]:
compact_factor_columns = [
    # Identifiers
    "month",
    "permno",
    "gvkey",
    "ticker",
    "company_name",
    "siccd",
    "primaryexch",

    # Point-in-time information
    "datadate",
    "accounting_available_date",
    "has_current_accounting_data",

    # Return and market data
    "monthly_return",
    "future_return_1m",
    "month_end_price",
    "month_end_market_cap",
    "market_cap_usd_millions",

    # Raw factor signals
    "book_to_market",
    "earnings_to_price",
    "momentum_12_1",
    "quality_profitability_raw",
    "quality_roe_raw",
    "quality_accruals_raw",
    "investment_growth_raw",
    "log_market_cap",
    "volatility_12m",

    # Standardized components
    "z_book_to_market",
    "z_earnings_to_price",
    "z_momentum_12_1",
    "z_quality_profitability",
    "z_quality_roe",
    "z_quality_accruals",
    "z_investment",
    "z_size",
    "z_low_volatility",

    # Factor families
    "factor_value",
    "factor_momentum",
    "factor_quality",
    "factor_investment",
    "factor_size",
    "factor_low_volatility",

    # Multi-factor score
    "factor_family_count",
    "multi_factor_score",
]

compact_factor_df = (
    factor_score_df
    .select(compact_factor_columns)
    .sort([
        "month",
        "permno",
    ])
)

compact_factor_df.write_csv(
    COMPACT_FACTOR_CSV_FILE
)

print("Compact factor CSV created successfully.")
print(f"Number of rows: {compact_factor_df.height:,}")
print(f"Number of columns: {compact_factor_df.width}")
print(f"CSV file: {COMPACT_FACTOR_CSV_FILE}")